In [1]:
# 03 — Feature Engineering: Attrition Model Features & Ablation Audit

import pandas as pd
import numpy as np
from pathlib import Path

PROC = Path(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\processed")
SNAPSHOT = pd.Timestamp('2025-12-31')

employees = pd.read_csv(PROC / 'employees_clean.csv', parse_dates=['hire_date', 'termination_date'])
compensation = pd.read_csv(PROC / 'compensation_clean.csv')
performance = pd.read_csv(PROC / 'performance_reviews_clean.csv', parse_dates=['review_date'])

comp_valid = compensation[compensation['is_valid_salary']]
dept_avg_salary = employees.merge(comp_valid, on='employee_id').groupby('department')['base_salary'].mean()
print("Department Benchmark Average Salaries ($):")
print(dept_avg_salary.round(0))


Department Benchmark Average Salaries ($):
department
Customer Success     93918.0
Engineering          96416.0
Finance             108135.0
HR                   94791.0
Marketing            96135.0
Operations           84115.0
Sales                93734.0
Name: base_salary, dtype: float64


In [2]:
## Cell 2: Engineer Latest Performance Score, Compa-Ratio, and Tenure Years

latest_perf = (
    performance[performance['is_valid_score']]
    .sort_values('review_date')
    .groupby('employee_id')
    .tail(1)[['employee_id', 'performance_score']]
)

df = employees.merge(comp_valid[['employee_id', 'base_salary']], on='employee_id', how='inner')
df = df.merge(latest_perf, on='employee_id', how='left')
df['performance_score'] = df['performance_score'].fillna(df['performance_score'].median())

df['compa_ratio'] = df.apply(
    lambda r: r['base_salary'] / dept_avg_salary.get(r['department'], r['base_salary']),
    axis=1
)
df['tenure_years'] = (df['termination_date'].fillna(SNAPSHOT) - df['hire_date']).dt.days / 365.25
df = df[df['is_valid_distance']].dropna(subset=['hire_date'])

print(f"{len(df):,} employees in the feature set")
df[['employee_id', 'department', 'tenure_years', 'compa_ratio', 'performance_score']].head(5)


3,228 employees in the feature set


,employee_id,department,tenure_years,compa_ratio,performance_score
0,EMP-11829,Operations,3.351129,0.795908,5.0
1,EMP-12626,Finance,3.112936,0.683865,1.0
2,EMP-11551,Customer Success,3.559206,0.578386,1.0
3,EMP-11236,Customer Success,2.839151,1.568437,3.0
4,EMP-12973,Engineering,0.613279,1.682531,1.0


In [3]:
## Cell 3: Save Engineered Feature Dataset

df.to_csv(PROC / 'attrition_features.csv', index=False)
print("Saved data/processed/attrition_features.csv")


Saved data/processed/attrition_features.csv


In [4]:
## Cell 4: Feature Ablation Stress-Test — Random Forest Without tenure_years

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

le_dept, le_gender, le_edu, le_marital = (LabelEncoder() for _ in range(4))
df['dept_enc'] = le_dept.fit_transform(df['department'])
df['gender_enc'] = le_gender.fit_transform(df['gender'])
df['edu_enc'] = le_edu.fit_transform(df['education_level'])
df['marital_enc'] = le_marital.fit_transform(df['marital_status'])

FEATURES_NO_TENURE = [
    'job_level', 'age', 'distance_from_home_miles', 'remote_flag',
    'compa_ratio', 'performance_score', 'dept_enc', 'gender_enc', 'edu_enc', 'marital_enc'
]
X = df[FEATURES_NO_TENURE]
y = (df['active_status'] == 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

m = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=15,
    random_state=42, class_weight='balanced'
)
m.fit(X_train, y_train)

auc_no_tenure = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
print(f"ROC-AUC without tenure_years: {auc_no_tenure:.3f}")
pd.Series(m.feature_importances_, index=FEATURES_NO_TENURE).sort_values(ascending=False)


ROC-AUC without tenure_years: 0.544


compa_ratio                 0.260245
distance_from_home_miles    0.207390
age                         0.155160
performance_score           0.101382
dept_enc                    0.074044
edu_enc                     0.047996
marital_enc                 0.045145
gender_enc                  0.042344
job_level                   0.035964
remote_flag                 0.030331
dtype: float64